[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pinecone-io/examples/blob/master/learn/generation/langchain/handbook/11-langchain-expression-language.ipynb) [![Open nbviewer](https://raw.githubusercontent.com/pinecone-io/examples/master/assets/nbviewer-shield.svg)](https://nbviewer.org/github/pinecone-io/examples/blob/master/learn/generation/langchain/handbook/11-langchain-expression-language.ipynb)

#### [LangChain Handbook](https://pinecone.io/learn/langchain)

# LangChain Expression Language (LCEL)

The **L**ang**C**hain **E**xpression **L**anguage (LCEL) is a abstraction of some interesting Python concepts into a format that enables a "minimalist" code layer for building chains of LangChain components.

LCEL comes with strong support for:

1. Superfast development of chains.
2. Advanced features such as streaming, async, parallel execution, and more.
3. Easy integration with LangSmith and LangServe.

In [1]:
# Use %pip (not !pip) so packages install into THIS notebook's kernel,
# not some other Python on your system. Restart the kernel after this runs.
%pip install -qU     langchain==0.3.25     langchain-community==0.3.25     langchain-google-genai==2.0.10     sniffio anyio pyparsing     docarray==0.39.1

# --- OpenRouter alternative (for reference) ---
# %pip install -qU langchain-openai==0.3.22 cohere==4.37

Note: you may need to restart the kernel to use updated packages.


## LCEL Syntax

To understand LCEL syntax let's first build a simple chain in typical Python syntax.

In [2]:
import os
from getpass import getpass

# We use a free Google Gemini LLM. Grab a free API key (no credit card) at
# https://aistudio.google.com/apikey, then input it when prompted (or set the
# GOOGLE_API_KEY environment variable).
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")     or getpass("Enter your Google API key: ")

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

# --- OpenRouter alternative (for reference) ---
# os.environ["OPENROUTER_API_KEY"] = os.getenv("OPENROUTER_API_KEY") or getpass("Enter your OpenRouter API key: ")
# OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

from langchain.prompts import ChatPromptTemplate
from langchain.schema.output_parser import StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI

prompt = ChatPromptTemplate.from_template(
    "Give me small report about {topic}"
)
model = ChatGoogleGenerativeAI(
    temperature=0,
    google_api_key=GOOGLE_API_KEY,
    model="gemini-2.5-flash",
)
output_parser = StrOutputParser()

# --- OpenRouter alternative (for reference) ---
# OpenRouter is OpenAI-compatible: use the ChatOpenAI class with a different base_url.
# Pick any ":free" model from https://openrouter.ai/models, e.g. "openai/gpt-oss-120b:free".
# from langchain_openai import ChatOpenAI
# model = ChatOpenAI(
#     model="openai/gpt-oss-120b:free",
#     temperature=0,
#     api_key=OPENROUTER_API_KEY,
#     base_url="https://openrouter.ai/api/v1",
# )

D:\Work\DEBI\Agents\.venv-langchain\Lib\site-packages\langchain_google_genai\chat_models.py:47: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  from google.generativeai.caching import CachedContent  # type: ignore[import]


In typical LangChain we would chain these together using an `LLMChain`:

In [3]:
from langchain.chains import LLMChain

chain = LLMChain(
    prompt=prompt,
    llm=model,
    output_parser=output_parser
)

# and run
out = chain.run(topic="Artificial Intelligence")
print(out)

C:\Users\pc\AppData\Local\Temp\ipykernel_25772\4093089756.py:3: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  chain = LLMChain(
C:\Users\pc\AppData\Local\Temp\ipykernel_25772\4093089756.py:10: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  out = chain.run(topic="Artificial Intelligence")


## Artificial Intelligence (AI): A Concise Report

**Introduction**
Artificial Intelligence (AI) refers to the simulation of human intelligence in machines that are programmed to think, learn, and solve problems like humans. It's a rapidly evolving field that aims to create systems capable of performing tasks that typically require human cognition, transforming industries and daily life.

**Core Concepts**
At its core, AI involves developing algorithms that enable computers to analyze data, recognize patterns, make decisions, and even learn from experience. Key distinctions include:

*   **Narrow AI (Weak AI):** Designed for specific tasks (e.g., Siri, recommendation systems, facial recognition). This is the AI we primarily interact with today.
*   **General AI (Strong AI):** Hypothetical AI with human-level cognitive abilities across various tasks, capable of understanding, learning, and applying intelligence to any problem. This has not yet been achieved.
*   **Machine Learning (ML):

Using LCEL the format is different, rather than relying on `Chains` we simple chain together each component using the pipe operator `|`:

In [4]:
lcel_chain = prompt | model | output_parser

# and run
out = lcel_chain.invoke({"topic": "Artificial Intelligence"})
print(out)

## Artificial Intelligence (AI): A Concise Report

**Introduction**
Artificial Intelligence (AI) refers to the simulation of human intelligence in machines that are programmed to think, learn, and solve problems like humans. It's a rapidly evolving field that aims to create systems capable of performing tasks that typically require human cognition, transforming industries and daily life.

**Core Concepts**
At its core, AI involves developing algorithms that enable computers to analyze data, recognize patterns, make decisions, and even learn from experience. Key distinctions include:

*   **Narrow AI (Weak AI):** Designed for specific tasks (e.g., Siri, recommendation systems, facial recognition). This is the AI we primarily interact with today.
*   **General AI (Strong AI):** Hypothetical AI with human-level cognitive abilities across various tasks, capable of understanding, learning, and applying intelligence to any problem. This has not yet been achieved.
*   **Machine Learning (ML):

Pretty cool, the way that this pipe operator is being used is that it is taking output from the function to the _left_ and feeding it into the function on the _right_.

## How the Pipe Operator Works

To really understand LCEL we can take a look at how this pipe operation works. We know it takes output from the _right_ and feeds it to the _left_ — but this isn't typical Python, so how is this implemented? Let's try creating our own version of this with some simple functions.

We will be using the `__or__` method within Python class objects. When we place two classes together like `chain = class_a | class_b` the Python interpreter will check if these classes contain this `__or__` method. If they do then our `|` code will be translated into `chain = class_a.__or__(class_b)`.

That means both of these patterns will return the same thing:

```python
# object approach
chain = class_a.__or__(class_b)
chain("some input")

# pipe approach
chain = class_a | class_b
chain("some input")

```

With that in mind, we can build a `Runnable` class that consumes a function and turns it into a function that can be chained with other functions using the pipe operator `|`.

In [5]:
class Runnable:
    def __init__(self, func):
        self.func = func

    def __or__(self, other):
        def chained_func(*args, **kwargs):
            # the other func consumes the result of this func
            return other(self.func(*args, **kwargs))
        return Runnable(chained_func)

    def __call__(self, *args, **kwargs):
        return self.func(*args, **kwargs)

Let's implement this to take the value `3`, add `5` (giving `8`), and multiply by `2` (giving `16`).

In [6]:
def add_five(x):
    return x + 5

def multiply_by_two(x):
    return x * 2

# wrap the functions with Runnable
add_five = Runnable(add_five)
multiply_by_two = Runnable(multiply_by_two)

# run them using the object approach
chain = add_five.__or__(multiply_by_two)
chain(3)  # should return 16

16

Using `__or__` directly we get the correct answer, now let's try using the pipe operator `|` to chain them together:

In [7]:
# chain the runnable functions together
chain = add_five | multiply_by_two

# invoke the chain
chain(3)  # we should return 16

16

Using either method we get the same response, at it's core this is what LCEL is doing, but there is more.

## LCEL Deep Dive

Now that we understand what this syntax is doing under the hood, let's explore it within the context of LCEL and see a few of the additional methods that LangChain has provided to maximize flexibility when working with LCEL.

### Runnables

When working with LCEL we may find that we need to modify the structure or values being passed between components — for this we can use _runnables_. Let's try:

In [8]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import DocArrayInMemorySearch

embedding = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=GOOGLE_API_KEY,
)

# --- OpenRouter / Cohere alternative (for reference) ---
# from langchain.embeddings import CohereEmbeddings
# embedding = CohereEmbeddings(
#     model="embed-english-light-v3.0",
#     cohere_api_key=COHERE_API_KEY,
# )

vecstore_a = DocArrayInMemorySearch.from_texts(
    ["half the info will be here", "James' birthday is the 7th December"],
    embedding=embedding
)
vecstore_b = DocArrayInMemorySearch.from_texts(
    ["and half here", "James was born in 1994"],
    embedding=embedding
)

First let's try passing a question through to a single one of these `vecstore` objects:

In [9]:
from langchain_core.runnables import (
    RunnableParallel,
    RunnablePassthrough
)

retriever_a = vecstore_a.as_retriever()
retriever_b = vecstore_b.as_retriever()

prompt_str = """Answer the question below using the context:

Context: {context}

Question: {question}

Answer: """
prompt = ChatPromptTemplate.from_template(prompt_str)

retrieval = RunnableParallel(
    {"context": retriever_a, "question": RunnablePassthrough()}
)

chain = retrieval | prompt | model | output_parser

In [10]:
out = chain.invoke("when was James born?")
print(out)

James was born on the 7th December.


Here we have used `RunnableParallel` to create two parallel streams of information, one for `"context"` that is information fed in from `retriever_a`, and another for `"question"` which is the _passthrough_ information, ie the information that is passed through from our `chain.invoke("when was James born?")` call.

Using this information the chain is close to answering the question but it doesn't have enough information, it is missing the information that we have stored in `retriever_b`. Fortunately, we can have multiple parallel information streams using the `RunnableParallel` object.

In [11]:
prompt_str = """Answer the question below using the context:

Context:
{context_a}
{context_b}

Question: {question}

Answer: """
prompt = ChatPromptTemplate.from_template(prompt_str)

retrieval = RunnableParallel(
    {
        "context_a": retriever_a, "context_b": retriever_b,
        "question": RunnablePassthrough()
    }
)

chain = retrieval | prompt | model | output_parser

In [12]:
out = chain.invoke("when was James born?")
print(out)

James was born on the 7th December 1994.


Now the `RunnableParallel` object is allowing us to search with `retriever_a` _and_ `retriever_b` in parallel, ie at the same time.

## Runnable Lambdas

The `RunnableLambda` is a LangChain abstraction that allows us to turn Python functions into pipe-compatible function, similar to the `Runnable` class we created near the beginning of this notebook.

Let's try it out with our earlier `add_five` and `multiply_by_two` functions.

In [13]:
from langchain_core.runnables import RunnableLambda

# wrap the functions with RunnableLambda
add_five = RunnableLambda(add_five)
multiply_by_two = RunnableLambda(multiply_by_two)

As with our earlier `Runnable` abstraction, we can use `|` operators to chain together our `RunnableLambda` abstractions.

In [14]:
chain = add_five | multiply_by_two

Unlike our `Runnable` abstraction, we cannot run the `RunnableLambda` chain by calling it directly, instead we must call `chain.invoke`:

In [15]:
chain.invoke(3)

16

As before, we can see the same answer. Naturally we can feed custom functions into our chains using this approach. Let's try a short chain and see where we might want to insert a custom function:

In [16]:
prompt_str = "Tell me an short fact about {topic}"
prompt = ChatPromptTemplate.from_template(prompt_str)

chain = prompt | model | output_parser

In [17]:
chain.invoke({"topic": "Artificial Intelligence"})

'Most AI today is considered "narrow AI," meaning it\'s designed to perform specific tasks, like recognizing faces or playing chess, rather than possessing general human-like intelligence.'

In [18]:
chain.invoke({"topic": "Artificial Intelligence"})

'The term "Artificial Intelligence" was coined in 1956 by computer scientist John McCarthy.'

The returned text always includes the initial `"Here's a short fact about ...\n\n"` — let's add a function to split on double newlines `"\n\n"` and only return the fact itself.

In [19]:
def extract_fact(x):
    if "\n\n" in x:
        return "\n".join(x.split("\n\n")[1:])
    else:
        return x
    
get_fact = RunnableLambda(extract_fact)

chain = prompt | model | output_parser | get_fact

In [20]:
chain.invoke({"topic": "Artificial Intelligence"})

'Most AI today is considered "narrow AI," meaning it\'s designed to perform specific tasks, like recognizing faces or playing chess, rather than possessing general human-like intelligence.'

In [21]:
chain.invoke({"topic": "Artificial Intelligence"})

'Most AI today is considered "narrow AI," meaning it\'s designed to perform specific tasks, like recognizing faces or playing chess, rather than possessing general human-like intelligence.'

Now we're getting well formatted outputs using our final custom `get_fact` function.

---

That covers the essentials you need to getting started and building with the **L**ang**C**hain **E**xpression **L**anguage (LCEL). With it, we can put together chains very easily — and the current focus of the LangChain team is on further LCEL development and support.

The pros and cons of LCEL are varied. Those that love it tend to focus on the minimalist code style, LCEL's support for streaming, parallel operations, and async, and LCEL's nice integration with LangChain's focus on chaining components together.

There are also people that are less fond of LCEL. Typically people will point to it being yet another abstraction on top of an already very abstract library, that the syntax is confusing, against [the Zen of Python](https://peps.python.org/pep-0020/), and requires too much effort to learn new (or uncommon) syntax.

Both viewpoints are entirely valid, LCEL is a very different approach — ofcourse there are major pros and major cons. In either case, if you're willing to spend some time learning the syntax, it allows us to develop very quickly, and with that in mind it's well worth learning.

---